In [20]:
import pandas as pd
fixed_entries = [ {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
 {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
{"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
  {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
{"question":"do i have any discounts?", "answer":"You can log into the portal to check.","keywords":"scholarship discount cashback", "category":"billing"},
{"question":"are the premises accessible?","answer":"We have ramps and lifts available alongside all stairs","keywords":"disability accessibility handicapped","category":"general"}
]
df=pd.DataFrame(fixed_entries)
print(df)

                       question  \
0        what is the annual fee   
1         how to reset password   
2   what are your working hours   
3         how can i pay the fee   
4      do i have any discounts?   
5  are the premises accessible?   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4              You can log into the portal to check.   
5  We have ramps and lifts available alongside al...   

                               keywords category  
0                 fee cost price charge  billing  
1                  password reset login  account  
2                hours timing open time  general  
3                   pay payment upi fee  billing  
4         scholarship discount cashback  billing  
5  disability accessibility handicapped  general  


In [21]:
import pandas as pd

def score_entries(query_string, entries_list):
    df = pd.DataFrame(entries_list)

    query_words = set(query_string.lower().split())

    scores = []
    for index, row in df.iterrows():
        entry_keywords = set(str(row['keywords']).lower().split())
        entry_question_words = set(str(row['question']).lower().split())

        keyword_matches = len(query_words.intersection(entry_keywords))
        question_matches = len(query_words.intersection(entry_question_words))

        score = keyword_matches * 2 + question_matches
        scores.append(score)

    df['confidence_score'] = scores

    ranked_entries = df[df['confidence_score'] > 0].sort_values(by='confidence_score', ascending=False)

    return ranked_entries

query = "how much does the annual fee cost?"
ranked_results = score_entries(query, fixed_entries)
print(ranked_results)

                       question  \
0        what is the annual fee   
3         how can i pay the fee   
1         how to reset password   
5  are the premises accessible?   

                                              answer  \
0                          The annual fee is Rs 500.   
3         You can pay via UPI, card, or net banking.   
1                   Go to Settings > Reset Password.   
5  We have ramps and lifts available alongside al...   

                               keywords category  confidence_score  
0                 fee cost price charge  billing                 5  
3                   pay payment upi fee  billing                 5  
1                  password reset login  account                 1  
5  disability accessibility handicapped  general                 1  


In [22]:
def same_category(category_name, data_list):
    df = pd.DataFrame(data_list)
    filtered_df = df[df['category'] == category_name]
    return filtered_df

print(same_category('billing', fixed_entries))

                   question                                      answer  \
0    what is the annual fee                   The annual fee is Rs 500.   
3     how can i pay the fee  You can pay via UPI, card, or net banking.   
4  do i have any discounts?       You can log into the portal to check.   

                        keywords category  
0          fee cost price charge  billing  
3            pay payment upi fee  billing  
4  scholarship discount cashback  billing  


In [23]:
df=pd.DataFrame(fixed_entries)
print(df.loc[1])
new_kw = str(input("Enter keyword: "))
df.loc[1, "keywords"] = df.loc[1, "keywords"] + " " + new_kw
print(df.loc[1])


question               how to reset password
answer      Go to Settings > Reset Password.
keywords                password reset login
category                             account
Name: 1, dtype: object
Enter keyword: sign in
question               how to reset password
answer      Go to Settings > Reset Password.
keywords        password reset login sign in
category                             account
Name: 1, dtype: object


In [24]:
df.to_csv('fixed_entries.csv', index=False)
print('DataFrame saved to fixed_entries.csv')

DataFrame saved to fixed_entries.csv


In [25]:
import pandas as pd
df = pd.DataFrame(fixed_entries)
result = df.groupby("category").count()
print(result)

          question  answer  keywords
category                            
account          1       1         1
billing          3       3         3
general          2       2         2


In [26]:
import pandas as pd

def score_entries_with_ties(query_string, entries_list):
    df = pd.DataFrame(entries_list)
    df['original_index'] = df.index

    query_words = set(query_string.lower().split())

    scores = []
    for index, row in df.iterrows():
        entry_keywords = set(str(row['keywords']).lower().split())
        entry_question_words = set(str(row['question']).lower().split())

        keyword_matches = len(query_words.intersection(entry_keywords))
        question_matches = len(query_words.intersection(entry_question_words))

        score = keyword_matches * 2 + question_matches
        scores.append(score)

    df['confidence_score'] = scores

    filtered_df = df[df['confidence_score'] > 0]

    if filtered_df.empty:
        return pd.DataFrame()

    max_score = filtered_df['confidence_score'].max()
    top_ranked_entries = filtered_df[filtered_df['confidence_score'] == max_score]

    return top_ranked_entries.sort_values(by=['confidence_score', 'original_index'], ascending=[False, True]).drop(columns=['original_index'])



query_tie = "fee"
print(f"--- Query: '{query_tie}' ---")
ranked_results_tie = score_entries_with_ties(query_tie, fixed_entries)
if not ranked_results_tie.empty:
    print("Top matching entries (with ties):")
    display(ranked_results_tie)
else:
    print("No matching entries found.")
print("\n" * 2)

query_no_tie = "reset password"
print(f"--- Query: '{query_no_tie}' ---")
ranked_results_no_tie = score_entries_with_ties(query_no_tie, fixed_entries)
if not ranked_results_no_tie.empty:
    print("Top matching entries (no tie expected):")
    display(ranked_results_no_tie)
else:
    print("No matching entries found.")


--- Query: 'fee' ---
Top matching entries (with ties):


,question,answer,keywords,category,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,3
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,3





--- Query: 'reset password' ---
Top matching entries (no tie expected):


,question,answer,keywords,category,confidence_score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,6
